# Beyond - Mixture of Experts

This notebook retrofits Module 09's transformer with a routed panel of expert FFNs and runs the model-card arithmetic for real: total versus active parameters, balanced versus collapsed routing, and a dense-versus-MoE comparison at near-matched active size. The required path uses the StoryLM-1M architecture on the 100MB TinyStories tier; an optional cell repeats the comparison with StoryLM-5M.

1. Read the lesson page (`docs/beyond/moe.md`).
2. Open this notebook with `./notebook.sh moe`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
import torch
import matplotlib.pyplot as plt

from g2c.artifacts import load_tokenized_corpus_artifact
from g2c.moe import MoEFeedForward, MoETransformerLM, Router
from g2c.transformer import TransformerLM

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")

## Exercise 1 — Collapse to dense

With one expert and `k = 1`, the MoE layer must reproduce Module 09's FFN *exactly* — the router has nobody to choose between and its single renormalized weight is 1.0. Verify it, then look at what an untrained router does with real choices.

In [ ]:
torch.manual_seed(0)
moe1 = MoEFeedForward(64, num_experts=1, top_k=1)
x = torch.randn(2, 5, 64)
gap = (moe1(x) - moe1.experts[0](x)).abs().max().item()
print(f"max |MoE(E=1,k=1) - dense FFN| = {gap:.2e}")

router = Router(64, num_experts=8, top_k=2)
weights, indices = router(torch.randn(6, 64))
print("untrained routing weights (each row sums to 1):")
for w, i in zip(weights, indices):
    print(f"  experts {i.tolist()}  weights {[round(v, 3) for v in w.tolist()]}")

In [ ]:
"Question: The router renormalizes the surviving top-k weights to sum to 1. What property of the layer's OUTPUT would silently degrade if you skipped the renormalization, and why doesn't any test of output *shape* catch it?"
"Answer: "

## Exercise 2 — Matched-active comparison

The fair fight: a dense model and an MoE model with near-matched active parameter counts (`k = 1` routed expert = one dense FFN of compute), where the MoE model holds 8× the FFN capacity in total. The small routers are the only meaningful active-parameter difference. Both models see the same TinyStories batches in the same order.

The corpus prep and the training loop are below — the loop is deliberately inline so you can see exactly where the balance loss joins the objective.

In [ ]:
CORPUS_NAME = "StoryLM-tinystories-100MB-v4096"
try:
    corpus = load_tokenized_corpus_artifact(CORPUS_NAME)
except FileNotFoundError as exc:
    raise RuntimeError(
        "TinyStories artifacts are missing. Run ./datasets.sh --tiny, "
        "then rerun this cell."
    ) from exc

pair = corpus.split(train_fraction=0.9, chunk_tokens=100_000, seed=12)
train_ids, val_ids = pair.train, pair.val
tokenizer = corpus.tokenizer
VOCAB = tokenizer.effective_vocab_size(4096)
print(f"corpus: {CORPUS_NAME}")
print(f"train tokens: {len(train_ids):,}; validation: {len(val_ids):,}")
print(f"effective vocabulary: {VOCAB:,}")

In [ ]:
from g2c.pretraining import get_lm_batch, lm_cross_entropy
from g2c.training import AdamW, clip_grad_norm_, cosine_with_warmup


def routing_utilization(model):
    counts = torch.zeros(model.num_experts)
    for block in model.blocks:
        counts += torch.bincount(
            block.moe_ffn.last_indices.reshape(-1).cpu(),
            minlength=model.num_experts,
        )
    return (counts / counts.sum()).tolist()


def train_lm(model, *, steps, max_lr, min_lr=3e-5, warmup_steps=100,
             batch_size=16, context=128, balance_coef=0.0,
             weight_decay=0.05, grad_clip=1.0, log_every=100,
             eval_iters=5, seed=0):
    gen = torch.Generator().manual_seed(seed)
    model.to(device)
    opt = AdamW(model.parameters(), max_lr, weight_decay=weight_decay)
    history = {"step": [], "train_loss": [], "val_loss": [],
               "utilization": []}
    for step in range(1, steps + 1):
        x, y = get_lm_batch(train_ids, batch_size, context, generator=gen)
        logits = model(x.to(device))
        loss = lm_cross_entropy(logits, y.to(device))
        if balance_coef:
            loss = loss + balance_coef * model.load_balancing_loss()
        opt.zero_grad()
        loss.backward()
        clip_grad_norm_(model.parameters(), grad_clip)
        opt.lr = cosine_with_warmup(
            step - 1, warmup_steps=warmup_steps, max_steps=steps,
            max_lr=max_lr, min_lr=min_lr,
        )
        opt.step()
        if step % log_every == 0:
            with torch.no_grad():
                losses = []
                for _ in range(eval_iters):
                    vx, vy = get_lm_batch(
                        val_ids, batch_size, context, generator=gen
                    )
                    losses.append(
                        lm_cross_entropy(
                            model(vx.to(device)), vy.to(device)
                        ).item()
                    )
                val_loss = sum(losses) / len(losses)
            history["step"].append(step)
            history["train_loss"].append(float(loss.detach()))
            history["val_loss"].append(val_loss)
            if isinstance(model, MoETransformerLM):
                history["utilization"].append(routing_utilization(model))
            print(f"step {step:>4}  val loss {val_loss:.3f}")
    return history

In [ ]:
storylm_1m = dict(vocab_size=VOCAB, embedding_dim=128, num_layers=4,
                  num_heads=4, max_seq_len=256, hidden_dim=512)
run = dict(steps=1_000, max_lr=3e-4, batch_size=16, context=128,
           log_every=100, seed=21)
BALANCE_COEF = 0.01

torch.manual_seed(1)
dense = TransformerLM(**storylm_1m)
torch.manual_seed(1)
moe = MoETransformerLM(**storylm_1m, num_experts=8, top_k=1)

n_dense = sum(p.numel() for p in dense.parameters())
print(f"dense:  {n_dense:,} params (total == active)")
print(f"moe:    {moe.total_parameter_count():,} total / "
      f"{moe.active_parameter_count():,} active")

print(f"router overhead: {moe.active_parameter_count() - n_dense:,} active parameters")

dense_history = train_lm(dense, **run)
moe_history = train_lm(moe, **run, balance_coef=BALANCE_COEF)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(dense_history["step"], dense_history["val_loss"], label="dense")
plt.plot(moe_history["step"], moe_history["val_loss"], label="MoE (8 experts, k=1)")
plt.xlabel("step"); plt.ylabel("val loss"); plt.legend()
plt.title("StoryLM-1M: near-matched active parameters")
plt.show()

In [ ]:
from g2c.sampling import generate


def print_story(label, model, seed):
    prompt = "Once upon a time"
    prompt_ids = torch.tensor(
        tokenizer.encode_with_vocab_size(prompt, VOCAB), dtype=torch.long
    )
    ids = generate(
        model, prompt_ids, 120, temperature=0.8, top_p=0.9,
        repetition_penalty=1.1,
        eos_id=tokenizer.special_to_id.get("<|endoftext|>"),
        generator=torch.Generator().manual_seed(seed),
    )
    print(f"--- {label}\n{tokenizer.decode(ids.tolist())}\n")


print_story("dense", dense, seed=7)
print_story("MoE", moe, seed=7)

In [ ]:
"Question: Your MoE model holds roughly 8x the FFN parameters of the dense model while activating one expert per token. Which parameter count is the better proxy for per-token compute, why is it not an exact latency or API-price formula, and why can the validation-loss gap still be modest at this scale?"
"Answer: "

## Exercise 3 — Top-1 utilization over training

The matched-active model uses renormalized `k=1`, so its selected weight is always 1. The language-modeling loss therefore gives the router no useful gradient; only the auxiliary loss moves it. Plot the recorded per-expert token fractions across all blocks and watch that balancing signal rather than interpreting this run as task-learned routing.

In [ ]:
util = torch.tensor(moe_history["utilization"])  # (n_logs, E)
plt.figure(figsize=(7, 4))
plt.stackplot(moe_history["step"], util.T, labels=[f"e{i}" for i in range(util.shape[1])])
plt.xlabel("step"); plt.ylabel("fraction of tokens")
plt.title(f"Top-1 utilization across blocks, balance_coef={BALANCE_COEF}")
plt.legend(loc="upper right", fontsize=7)
plt.show()

## Exercise 4 — Ablate the balance loss with k=2

Use `k=2` for this routing-dynamics experiment. Two surviving experts retain differentiable relative weights, so the language-modeling loss can now train the router. Run coefficient `0`, the calibrated default, and `100×` default with identical initialization, batches, and step budgets. A short toy run may show weak rather than dramatic concentration; report the plots you get, not the plot the theory told you to expect.

In [ ]:
ablation_histories = {}
ablation_models = {}
for coef in (0.0, BALANCE_COEF, 100 * BALANCE_COEF):
    torch.manual_seed(1)
    model = MoETransformerLM(
        **storylm_1m, num_experts=8, top_k=2
    )
    print(f"--- balance_coef={coef:g}")
    ablation_histories[coef] = train_lm(
        model, **run, balance_coef=coef
    )
    ablation_models[coef] = model

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, coef in zip(axes, (0.0, BALANCE_COEF, 100 * BALANCE_COEF)):
    history = ablation_histories[coef]
    util = torch.tensor(history["utilization"])
    ax.stackplot(
        history["step"], util.T,
        labels=[f"e{i}" for i in range(util.shape[1])],
    )
    ax.set_title(f"balance_coef={coef:g}")
    ax.set_xlabel("step")
axes[0].set_ylabel("fraction of assignments")
axes[-1].legend(loc="upper right", fontsize=7)
fig.tight_layout()
plt.show()

In [ ]:
"Question: Describe what your k=2 zero/default/100x utilization plots actually showed. Why does this sweep use k=2 instead of the headline model's renormalized k=1? Then explain both pressures: why can a small early advantage compound without balancing, and what useful routing behavior can be lost when the balancing term dominates the language-modeling loss?"
"Answer: "

## Exercise 5 — Sweep k (optional)

More active experts per token means more compute per token. Reuse the balanced `k=1` headline run and balanced `k=2` ablation, train only the remaining `k=4` model, and compare validation loss. Is the extra active compute buying loss at this scale?

In [ ]:
k_histories = {
    1: moe_history,
    2: ablation_histories[BALANCE_COEF],
}
for k in (4,):
    torch.manual_seed(1)
    model_k = MoETransformerLM(
        **storylm_1m, num_experts=8, top_k=k
    )
    print(f"--- top_k={k}  "
          f"(active {model_k.active_parameter_count():,})")
    k_histories[k] = train_lm(
        model_k, **run, balance_coef=BALANCE_COEF
    )

plt.figure(figsize=(7, 4))
for k, h in k_histories.items():
    plt.plot(h["step"], h["val_loss"], label=f"top_k={k}")
plt.xlabel("step"); plt.ylabel("val loss"); plt.legend(); plt.show()

## Exercise 6 — StoryLM-5M rerun (optional)

Run the next cell only if you want the larger comparison. It repeats the same `E=8, k=1` design with Module 10's StoryLM-5M architecture. The dense model is about 5.86M parameters; the MoE is about 27.94M total / 5.87M active. These runs are substantially longer than the required StoryLM-1M path.

In [ ]:
storylm_5m = dict(vocab_size=VOCAB, embedding_dim=256, num_layers=6,
                  num_heads=8, max_seq_len=256, hidden_dim=1024)
run_5m = dict(steps=2_000, max_lr=3e-4, batch_size=8, context=128,
              warmup_steps=200, log_every=200, seed=51)

torch.manual_seed(5)
dense_5m = TransformerLM(**storylm_5m)
torch.manual_seed(5)
moe_5m = MoETransformerLM(**storylm_5m, num_experts=8, top_k=1)
print(f"dense: {sum(p.numel() for p in dense_5m.parameters()):,}")
print(f"MoE: {moe_5m.total_parameter_count():,} total / "
      f"{moe_5m.active_parameter_count():,} active")
dense_5m_history = train_lm(dense_5m, **run_5m)
moe_5m_history = train_lm(
    moe_5m, **run_5m, balance_coef=BALANCE_COEF
)

plt.figure(figsize=(7, 4))
plt.plot(dense_5m_history["step"], dense_5m_history["val_loss"],
         label="dense 5M")
plt.plot(moe_5m_history["step"], moe_5m_history["val_loss"],
         label="MoE 5M")
plt.xlabel("step"); plt.ylabel("val loss"); plt.legend(); plt.show()
print_story("dense 5M", dense_5m, seed=9)
print_story("MoE 5M", moe_5m, seed=9)

## Exercise 7 — Specialization probe (optional)

Which decoded TinyStories tokens go to which expert? Route one batch through the balanced `k=2` StoryLM-1M model and bucket tokens by their primary (highest-weight) expert in the last block. Unlike the `k=1` headline router, this router received a language-modeling gradient through its relative combination weights. At toy scale the structure may still be noisy; an honest null result is useful evidence.

In [ ]:
from collections import Counter

x, _ = get_lm_batch(
    train_ids, 32, 128, generator=torch.Generator().manual_seed(2)
)
probe_moe = ablation_models[BALANCE_COEF]
with torch.no_grad():
    probe_moe(x.to(device))
assigned = probe_moe.blocks[-1].moe_ffn.last_indices[:, 0].reshape(32, 128).cpu()
for e in range(probe_moe.num_experts):
    selected = x[assigned == e].tolist()
    top = [
        (repr(tokenizer.decode([token_id])), count)
        for token_id, count in Counter(selected).most_common(8)
    ]
    share = len(selected) / assigned.numel()
    print(f"expert {e} ({share:5.1%}): {top}")

In [ ]:
"Question: From your balanced k=2 specialization probe: name one primary-expert bucket that looks interpretable and one that does not, or report that all buckets look noisy. Explain why per-token routing can specialize by token role or local context, and why a StoryLM-1M-scale run may not produce clean expert roles."
"Answer: "

When complete, ask a coding agent to grade your notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.